# Breast Cancer Diagnosis using Machine Learning

**Project by:** Charitha Piyumal  
**Date:** December 12, 2025

## 1. Introduction
Breast cancer is one of the most common cancers worldwide, and early detection is critical for successful treatment. This project aims to develop a machine learning model capable of accurately diagnosing breast tumors as either **Malignant (Cancerous)** or **Benign (Non-cancerous)** based on cell nucleus features derived from fine needle aspirate (FNA) images.

### Project Goals:
1.  **Analyze** the Breast Cancer Wisconsin (Diagnostic) Dataset.
2.  **Preprocess** the data by scaling features and handling potential issues.
3.  **Train** multiple classification models:
    * Logistic Regression
    * Support Vector Machine (SVM)
    * Artificial Neural Network (MLP)
4.  **Optimize** these models using Hyperparameter Tuning (GridSearch).
5.  **Evaluate** the best model to achieve maximum diagnostic accuracy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Machine Learning Libraries
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Configuration
warnings.filterwarnings('ignore') # Clean up output
plt.style.use('seaborn-v0_8')

# Setup Folders for saving results
os.makedirs('../data', exist_ok=True)
os.makedirs('../images', exist_ok=True)

print("✅ Libraries loaded and environment configured.")

## 2. Dataset Loading and Exploration
We utilize the **Breast Cancer Wisconsin (Diagnostic) Dataset** provided by `scikit-learn`. 
* **Samples:** 569
* **Features:** 30 numeric attributes (e.g., Radius, Texture, Smoothness)
* **Target:** 0 (Malignant) / 1 (Benign)

In [ ]:
# 1. Load the dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

# 2. Save raw data for the report
df.to_csv('../data/breast_cancer_data.csv', index=False)

# 3. Display dataset structure
print(f"Dataset Shape: {df.shape}")
print("\n--- First 5 Rows ---")
display(df.head())

# 4. Check for missing values
print("\n--- Missing Values ---")
print(df.isnull().sum().sum())

# Display information about the DataFrame
display(df.info())

#Discribe the data
print(df.describe())


## 3. Data Visualization
Exploratory Data Analysis (EDA) is performed to understand the distribution of the target variable and the relationships between features.

### 3.1 Target Distribution
We visualize the balance of the dataset using a count plot. This helps identify if the classes (Malignant vs. Benign) are imbalanced, which could bias model training.
* **0 (Malignant)**: Cancerous tumors.
* **1 (Benign)**: Non-cancerous tumors.

### 3.2 Correlation Heatmap
A correlation matrix is generated for the first 10 features to analyze multicollinearity. High positive values (closer to 1.0) indicate strong correlation, suggesting that some features might provide redundant information (e.g., radius vs. area).

In [ ]:
# --- Visualization ---

# 1. Target Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df)
plt.title('Class Distribution (0=Malignant, 1=Benign)')
plt.xlabel('Diagnosis')
plt.ylabel('Count')

# SAVE to images folder
plt.savefig('../images/distribution.png') 
plt.show()

# 2. Correlation Heatmap
plt.figure(figsize=(12, 10))
subset = df.iloc[:, :10] # First 10 features
sns.heatmap(subset.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Heatmap (Top 10 Features)')

# SAVE to images folder
plt.savefig('../images/heatmap.png')
plt.show()

## 4. Data Preprocessing
Machine learning models, especially Neural Networks and SVMs, perform better when input features are on the same scale.
1.  **Splitting:** We split the data into **Training (80%)** and **Testing (20%)** sets.
2.  **Scaling:** We use `StandardScaler` to normalize features (Mean = 0, Variance = 1).



In [ ]:
# 1. Define Features (X) and Target (y)
X = df.drop('target', axis=1)
y = df['target']

# 2. Split Data (80% Train, 20% Test)
# random_state=42 ensures we get the same split every time (reproducibility)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Set Shape: {X_train_scaled.shape}")
print(f"Testing Set Shape:  {X_test_scaled.shape}")



In [ ]:
# --- Visualization of Split and Scaling ---

# Visualize the Split Ratio
labels = ['Training Set', 'Testing Set']
sizes = [len(X_train), len(X_test)]
colors = ['#66b3ff', '#ff9999']

plt.figure(figsize=(6, 6))
plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90, explode=(0.1, 0))
plt.title('Train-Test Split Ratio')
plt.savefig('../images/split_ratio.png')
plt.show()


In [ ]:
# Visualize Scaling Effect (Before vs After)

# We pick one feature (e.g., 'mean area') to show the difference
feature_index = 3 # 'mean area' is typically index 3
feature_name = data.feature_names[feature_index]

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# Before Scaling
sns.histplot(X_train.iloc[:, feature_index], kde=True, ax=ax[0], color='orange')
ax[0].set_title(f'Before Scaling: {feature_name}')
ax[0].set_xlabel('Original Value')

# After Scaling
sns.histplot(X_train_scaled[:, feature_index], kde=True, ax=ax[1], color='green')
ax[1].set_title(f'After Scaling: {feature_name}')
ax[1].set_xlabel('Scaled Value (Z-Score)')

plt.tight_layout()
plt.savefig('../images/scaling_effect.png')
plt.show()

### 4.1 Class Distribution
This visualization shows the count of **Benign** vs. **Malignant** samples in the dataset.
* **0 (Malignant):** Cancerous cases.
* **1 (Benign):** Non-cancerous cases.

Checking this distribution is important to see if the dataset is balanced. An imbalanced dataset (e.g., 90% Benign, 10% Malignant) might require special handling, like resampling, to ensure the model learns to detect cancer effectively.

In [ ]:
# --- Visualization: Class Distribution ---

plt.figure(figsize=(7, 5))
# Using 'viridis' palette for better contrast
ax = sns.countplot(x='target', data=df, palette='viridis')

# Add labels
plt.title('Class Distribution: Benign vs. Malignant', fontsize=14)
plt.xlabel('Diagnosis (0=Malignant, 1=Benign)', fontsize=12)
plt.ylabel('Count of Samples', fontsize=12)

# Add the actual numbers on top of the bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', fontsize=11, color='black', xytext=(0, 5),
                textcoords='offset points')

# Save and Show
plt.savefig('../images/class_distribution.png')
plt.show()

### 4.2 Correlation Heatmap
We generate a correlation matrix to analyze relationships between features. In this heatmap, we specifically analyze the "Mean" features (the first 10 columns) to check for **multicollinearity**.

**Key Insight:**
There are extremely strong positive correlations (approaching 1.0) between the size-based features:
* **Radius vs. Perimeter**
* **Radius vs. Area**

This is expected since these measures are geometrically related. However, for machine learning models like Logistic Regression, providing multiple features that carry the exact same information can be redundant.

In [ ]:
# --- Visualization: Correlation Heatmap ---

plt.figure(figsize=(10, 8))

# We select the first 10 features (the 'mean' values) to keep the chart readable
# This includes the size features: mean radius, mean perimeter, mean area
subset_features = df.iloc[:, :10]
correlation_matrix = subset_features.corr()

# Plotting
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap: Mean Features', fontsize=14)

# Save and Show
plt.savefig('../images/heatmap.png')
plt.show()

## 5. Model Training and Hyperparameter Tuning
We will train three models using `GridSearchCV`. This technique tests multiple combinations of parameters to find the absolute best configuration.

**Models:**
1.  **Logistic Regression:** A strong baseline for binary classification.
2.  **Support Vector Machine (SVM):** Excellent for high-dimensional data.
3.  **Neural Network (MLP):** Capable of capturing complex non-linear patterns.

*> Note: We have expanded the parameter grid for the Neural Network to improve accuracy.*

In [ ]:
import time

# --- 1. Logistic Regression ---
print("Training Logistic Regression...")
lr_params = {'C': [0.1, 1, 10, 100], 'solver': ['liblinear', 'lbfgs']}
lr_grid = GridSearchCV(LogisticRegression(max_iter=5000), lr_params, cv=5, n_jobs=-1)
lr_grid.fit(X_train_scaled, y_train)
print(f"Best LR Params: {lr_grid.best_params_}")

# --- 2. Support Vector Machine (SVM) ---
print("\nTraining SVM...")
svm_params = {
    'C': [0.1, 1, 10, 100], 
    'kernel': ['linear', 'rbf'], 
    'gamma': ['scale', 'auto']
}
svm_grid = GridSearchCV(SVC(probability=True), svm_params, cv=5, n_jobs=-1)
svm_grid.fit(X_train_scaled, y_train)
print(f"Best SVM Params: {svm_grid.best_params_}")

# --- 3. Neural Network (Extended Grid for High Accuracy) ---
print("\nTraining Neural Network (This may take a moment)...")
nn_params = {
    'hidden_layer_sizes': [(50,), (100,), (50, 50), (100, 50)], # Added (100, 50)
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],             # Added solver tuning 
    'alpha': [0.0001, 0.001, 0.01],        # Added regularization tuning
    'learning_rate': ['constant', 'adaptive'], # Added learning rate tuning
    'max_iter': [2000]
}
nn_grid = GridSearchCV(MLPClassifier(random_state=42), nn_params, cv=3, n_jobs=-1)
nn_grid.fit(X_train_scaled, y_train)
print(f"Best NN Params: {nn_grid.best_params_}")

Training Logistic Regression...
Best LR Params: {'C': 0.1, 'solver': 'liblinear'}

Training SVM...
Best SVM Params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}

Training Neural Network (This may take a moment)...
Best NN Params: {'activation': 'tanh', 'alpha': 0.0001, 'hidden_layer_sizes': (50,), 'learning_rate': 'constant', 'max_iter': 2000, 'solver': 'sgd'}


## 6. Model Evaluation
We evaluate the best version of each model on the **Test Set** (data the models have never seen before).
Key Metrics:
* **Accuracy:** Overall correctness.
* **Recall:** (Crucial for medical diagnosis) How many actual cancer cases did we catch?
* **Precision:** How many predicted cancer cases were actually cancer?

In [ ]:
def evaluate_model(model, X, y, name):
    y_pred = model.predict(X)
    
    # Calculate Scores
    acc = accuracy_score(y, y_pred)
    prec = precision_score(y, y_pred)
    rec = recall_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    
    print(f"\n--- {name} Performance ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    
    # Confusion Matrix
    plt.figure(figsize=(5, 4))
    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.savefig(f"../images/cm_{name.replace(' ', '_')}.png") # Save for report
    plt.show()

# Run Evaluation
evaluate_model(lr_grid.best_estimator_, X_test_scaled, y_test, "Logistic Regression")
evaluate_model(svm_grid.best_estimator_, X_test_scaled, y_test, "SVM")
evaluate_model(nn_grid.best_estimator_, X_test_scaled, y_test, "Neural Network")



## 7. Conclusion

This analysis successfully implemented and compared three machine learning models for breast cancer diagnosis: Logistic Regression, Support Vector Machine (SVM), and an Artificial Neural Network (MLP).

### Key Findings:
* **Best Performing Model:** **Logistic Regression** achieved the highest overall performance with an **Accuracy of 99.12%**.
* **Clinical Reliability (Recall):** Both **Logistic Regression** and **SVM** achieved a perfect **Recall score of 1.0000**. This is the most critical metric for medical diagnosis, as it means these models successfully identified **100% of the Malignant (cancerous)** cases in the test set, with zero False Negatives.
* **Neural Network Performance:** While the Neural Network performed very well (98.25% Accuracy), it slightly underperformed compared to the simpler Logistic Regression model on this specific test set.

### Summary of Metrics:
| Model | Accuracy | Precision | Recall | F1 Score |
| :--- | :--- | :--- | :--- | :--- |
| **Logistic Regression** | **0.9912** | **0.9861** | **1.0000** | **0.9930** |
| SVM | 0.9825 | 0.9726 | **1.0000** | 0.9861 |
| Neural Network | 0.9825 | 0.9859 | 0.9859 | 0.9859 |

**Final Recommendation:** Based on these results, **Logistic Regression** is the recommended model for this dataset. It offers the perfect balance of high accuracy and, most importantly, perfect sensitivity (Recall) to ensure no cancer cases are missed.